In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [17]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-25 16:21:44--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-25 16:21:44 (21.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('data/input.txt','r', encoding = 'utf-8') as f:
    text = f.read()


In [3]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f"Vocab size is {vocab_size}")


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size is 65


### Encoder & Decoder

In [5]:
## Encoder 
character_mapping = {ch:i for i,ch in enumerate(chars)}
encoder = lambda s:[character_mapping[c] for c in s]

## Decoder
index_mapping = {i:ch for i,ch in enumerate(chars)}
decoder = lambda l:''.join([index_mapping[i] for i in l])


In [6]:
print(f"Original text: {text[:15]}")
encoded_message = encoder(text[:15])
print(f"Encoded message : {encoded_message}")
print(f"Decoded message : {decoder(encoded_message)}")

Original text: First Citizen:

Encoded message : [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0]
Decoded message : First Citizen:



In [7]:
input_data = torch.tensor(encoder(text), dtype= torch.int64)
print(input_data.shape)
print(input_data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
train_size = int(0.9*len(input_data))
train_data = input_data[:train_size]
val_data = input_data[train_size:]


In [9]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"Target for context {context} is {target}")


Target for context tensor([18]) is 47
Target for context tensor([18, 47]) is 56
Target for context tensor([18, 47, 56]) is 57
Target for context tensor([18, 47, 56, 57]) is 58
Target for context tensor([18, 47, 56, 57, 58]) is 1
Target for context tensor([18, 47, 56, 57, 58,  1]) is 15
Target for context tensor([18, 47, 56, 57, 58,  1, 15]) is 47
Target for context tensor([18, 47, 56, 57, 58,  1, 15, 47]) is 58


In [10]:
len(train_data) - block_size

1003846

In [11]:
torch.manual_seed(1337)
batch_size = 4 # parallel batches processed
block_size = 8 # context length

def get_batch(split):
    """ Function to get batch data"""
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)- block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+1+block_size] for i in ix])
    return x,y

xb, yb = get_batch('train')
print(xb.shape)
print("Input :")
print(xb)
print("Output :")
print(yb)





torch.Size([4, 8])
Input :
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Output :
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [12]:
layer = nn.Embedding(65,65)

layer.weight[0].shape

torch.Size([65])

### Version 1 : Bigram Model

In [13]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding_layer = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.embedding_layer(idx)
        if targets is None:
            loss = None

        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_token):
        """ Generate max new tokens basis the initial idx"""
        for _ in range(max_new_token):
            logits, loss = self(idx)
            logits = logits[:,-1,:] # (B,C)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1) #(B, T+1)
        return idx


In [14]:
model = BigramModel(vocab_size=65)
logits, loss = model(idx = xb, targets = yb)
print(loss)
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))

tensor(4.7288, grad_fn=<NllLossBackward0>)

DmCLUzAteq'3ggtzCET?XYKbA?W&!rFtYub!SlC.w BHf;I?KFyXOWIWhh3Qh; ZibtgnwNup?rYjEXYuiWrKBPxrUUCzyajQ.!b


In [15]:
## Training loop

optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
batch_size = 32

for _ in range(10000): # Number of iterations
    xb, yb = get_batch(batch_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


2.3323283195495605


In [16]:
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))


FENZAPlucor che t thendouiletheesco'imin?
DERUMisoupll nn o:
Be; aloricr
ARI t:

Fot ms, sth ht hon 


In [17]:
torch.arange(8)

tensor([0, 1, 2, 3, 4, 5, 6, 7])

### Version 2: Self Attention

In [23]:
n_embed = 32
class Head(nn.Module):
    def __init__(self, head_size): ## head_size = d_k
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size,bias = False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) ##(B,T,C)
        q = self.query(x) ##(B,T,C)
        v = self.value(x) ##(B,T,C)

        ## Compute attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5 ## (B,T,C) * (B,C,T) --> (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        out = wei @ v
        return out
    

        

In [28]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.pos_embedding = nn.Embedding(block_size, n_embed)
        self.sa = Head(n_embed)
        self.lm = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets = None):
        B,T = idx.shape
        tok_embed = self.token_embedding(idx) #(B,T,C)
        pos_embed = self.pos_embedding(torch.arange(T)) ## (T,C)
        x = tok_embed + pos_embed
        x = self.sa(x) ##(B,T,C)
        logits = self.lm(x) ## (B,T,vocab_size)

        if targets == None:
            loss = None

        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_token):
        for _ in range(max_new_token):
            idx_cond = idx[:, -block_size:] # restrict idx to (B,T)
            logits, loss = self(idx_cond)
            ## Get the last logit
            logit = logits[:, -1, :]
            ## Get the prob of last logit's C dimension
            probs = F.softmax(logit, dim = -1)
            ## Pick up the most likely next token
            idx_next = torch.multinomial(probs, num_samples=1) ##(B,1)
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx
    



In [29]:

model = Model()
logits, loss = model(idx = xb, targets = yb)
print(loss)
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))

tensor(4.2249, grad_fn=<NllLossBackward0>)

p3I-bOOJX?ihBEbegKmpHWtPoovGopDQB&WyWphU?vFqe KVeQassL-3fFpVOD'chcFXsAqXzYDZgM.H?KZZtftLml&ZEZCqmLvn


In [30]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
batch_size = 32

for _ in range(10000): # Number of iterations
    xb, yb = get_batch(batch_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.296598196029663


In [32]:
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))


Kan'depal ge wheast.

Torod ps, me art ayt brepir cikeeres iarile tisoy Sot ee
N; aus sind.

Whithe



### Version 3: Multi-Head Attention

In [42]:
class MultiHead(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim = -1)
    


In [43]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.pos_embedding = nn.Embedding(block_size, n_embed)
        self.multi_head = MultiHead(n_head = 4, head_size = n_embed//4)
        self.lm = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets = None):
        B,T = idx.shape
        tok_embed = self.token_embedding(idx) #(B,T,C)
        pos_embed = self.pos_embedding(torch.arange(T)) ## (T,C)
        x = tok_embed + pos_embed
        x = self.multi_head(x) ##(B,T,C)
        logits = self.lm(x) ## (B,T,vocab_size)

        if targets == None:
            loss = None

        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_token):
        for _ in range(max_new_token):
            idx_cond = idx[:, -block_size:] # restrict idx to (B,T)
            logits, loss = self(idx_cond)
            ## Get the last logit
            logit = logits[:, -1, :]
            ## Get the prob of last logit's C dimension
            probs = F.softmax(logit, dim = -1)
            ## Pick up the most likely next token
            idx_next = torch.multinomial(probs, num_samples=1) ##(B,1)
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx
    



In [45]:

model = Model()
logits, loss = model(idx = xb, targets = yb)
print(loss)
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))

tensor(4.1888, grad_fn=<NllLossBackward0>)

aOHX?bddZBfS'OIUZGHDMpKoLsDgv.njhn:gD
WIEAll OaQrSfk&MpqxSqgtq!CCh.NrKiJ;R'ObtNZFYbXPo?q.AV
DeyB:vMV


In [46]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
batch_size = 32

for _ in range(10000): # Number of iterations
    xb, yb = get_batch(batch_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

1.9162230491638184


In [47]:
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))


BITENSIO:
ot? miforin!
DVINA:
Pery hathuppreade re, nughtap ow yous:
Fandead nas, a?

HORO:
I in mey


### Version 4: With the FeedForward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super()._init__()
        self.layer = nn.Linear(n_embed, n_embed)
        self.relu = nn.ReLU()
        

    def forward(self, x):
        return self.relu(self.layer(x))
    
        

In [48]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.pos_embedding = nn.Embedding(block_size, n_embed)
        self.multi_head = MultiHead(n_head = 4, head_size = n_embed//4)
        self.ff = FeedForward(n_embed)
        self.lm = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets = None):
        B,T = idx.shape
        tok_embed = self.token_embedding(idx) #(B,T,C)
        pos_embed = self.pos_embedding(torch.arange(T)) ## (T,C)
        x = tok_embed + pos_embed
        x = self.multi_head(x) ##(B,T,C)
        x = self.ff(x)
        logits = self.lm(x) ## (B,T,vocab_size)

        if targets == None:
            loss = None

        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_token):
        for _ in range(max_new_token):
            idx_cond = idx[:, -block_size:] # restrict idx to (B,T)
            logits, loss = self(idx_cond)
            ## Get the last logit
            logit = logits[:, -1, :]
            ## Get the prob of last logit's C dimension
            probs = F.softmax(logit, dim = -1)
            ## Pick up the most likely next token
            idx_next = torch.multinomial(probs, num_samples=1) ##(B,1)
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx
    



In [49]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
batch_size = 32

for _ in range(10000): # Number of iterations
    xb, yb = get_batch(batch_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.0696358680725098


In [50]:
print(decoder(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_token=100)[0].tolist()))


Godeve, ther. Sagalsaurain; iblenent?

TRANIO:

Ay
blintre't.

PETRUCHIO:
Not the pinech ling nign t


### Version 5 : With Residual Connections, Blocks and layer norm

In [83]:
block_size = 256
batch_size = 64
learning_rate = 3e-4
dropout = 0.2
n_embed = 384
n_head = 6
n_layer = 6

In [84]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.layer = nn.Linear(n_embed, 4* n_embed) ## 4*n_embed since it's so in the paper
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(4*n_embed, n_embed)
        

    def forward(self, x):
        return self.layer2(self.relu(self.layer(x)))

In [85]:
class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
        self.attention = MultiHead(n_head, head_size)
        self.ff = FeedForward(n_embed)
        self.proj = nn.Linear(n_embed, n_embed) ## To go back into the main path - due to residual connection

    def forward(self,x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        x = self.proj(x)
        return x

In [86]:
class MultiHead(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(head_size*n_head, n_embed)

    def forward(self, x):
        out =  torch.cat([h(x) for h in self.heads], dim = -1)
        return self.proj(out)

In [87]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.pos_embedding = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(
            Block(n_embed, n_head=4),
            Block(n_embed, n_head=4),
            Block(n_embed, n_head=4),
            nn.LayerNorm(n_embed)
            )
        self.lm = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets = None):
        B,T = idx.shape
        tok_embed = self.token_embedding(idx) #(B,T,C)
        pos_embed = self.pos_embedding(torch.arange(T)) ## (T,C)

        x = tok_embed + pos_embed
        x = self.blocks(x)
        logits = self.lm(x) ## (B,T,vocab_size)

        if targets == None:
            loss = None

        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_token):
        for _ in range(max_new_token):
            idx_cond = idx[:, -block_size:] # restrict idx to (B,T)
            logits, loss = self(idx_cond)
            ## Get the last logit
            logit = logits[:, -1, :]
            ## Get the prob of last logit's C dimension
            probs = F.softmax(logit, dim = -1)
            ## Pick up the most likely next token
            idx_next = torch.multinomial(probs, num_samples=1) ##(B,1)
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx
    



In [88]:
class Head(nn.Module):
    def __init__(self, head_size): ## head_size = d_k
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size,bias = False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))
        self.droput = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) ##(B,T,C)
        q = self.query(x) ##(B,T,C)
        v = self.value(x) ##(B,T,C)

        ## Compute attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5 ## (B,T,C) * (B,C,T) --> (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        wei= self.droput(wei)
        out = wei @ v
        return out

In [ ]:
model = Model()
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)
batch_size = 32

for _ in range(1000): # Number of iterations
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

In [ ]:
print(decoder(model.generate(idx=xb, max_new_token=100)[0].tolist()))

st it shy nobe. Pedr!
soughe thou the heevess foome with koortts be, ris stis merime corey vester lold to or


In [64]:
decoder(xb[0].tolist())

'st it sh'